learning how to flow

In [1]:
%load_ext autoreload
%autoreload 2

# IMPORTS
import time
import tqdm
import math

import torch
from torch import nn
import numpy as np
# import torchdyn
from torchdyn.core import NeuralODE 


from data import * #only here bc you can plot the data for the main notebook, but is generallly used in the training loop inside flow_matching.py
from dpd import *
from model_architecture import * 
from transformer import *
# from utils import *
from flow_matching import * 
from discrete_fm import *


In [2]:
####### DEVICE SETUP #######
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("device is", device)

device is mps


### Data & Sampling


**data.py**

Sample from the distributions (Cat or Moon)  
Data sits on -4 to 4.0 grid in the X-Y axis space  

**def sample_cats(N=1500):**  
&emsp;Samples N points from the cat distribution.  
&emsp;Return:  
&emsp;&emsp;numpy array of each xy coordinate in the sampled dist

**def sample_moons(N=1500):**  
&emsp;Uses the torchdfyn module to generate N samples of the moon distribution  
&emsp;Return:  
&emsp;&emsp;torch.tensor of each xy coordinate in the sampled dist. 

**def plot_distribution(sampled_points[list], dist_name[list],color[list]):**  
&emsp;Plots the sampled points, dist_name and color are lists of strings  

In [3]:
# ############ UNCOMMENT TO PLOT DISTRIBUTIONS ############
# #sample and plot cats
# sampled_cats = sample_cats(1500)
# # plot_distribution([sampled_cats], ["Cats"], ["gold"])
# #sample and plot moons
# sampled_moons = sample_moons(1500)
# # plot_distribution([sampled_moons], ["Moon"], ["steelblue"])
# #plot both together
# plot_distribution([sampled_moons, sampled_cats], ["Moon", "Cats"], ["steelblue", "gold"])

**dpd.py**

Generates some type of source distribution (symmetric or binomial) and applies a discrete process to it (poisson) to generate the target distribution

In [ ]:
SEED = 10
N = 1500
rng = np.random.default_rng(SEED)
CENTERS = [20]
N_MEASURES = 2
#initialize the data object using the list of centers
spec = make_measure_specs(centers=CENTERS, seed=SEED)

#initialize the data object using the list of centers
# spec = make_measure_specs(n_measures=N_MEASURES, seed=SEED)

#create the source and target according to the spec parameters, then plot
for s in spec:
    #processed is the timesteps of the discrete process
    processed = generate_target_dist(spec=s, sample_size=N, rng=rng)

    # plot_dpd(processed, sample_type=s.sample_type, process_type=s.process_type, center=s.center)

    animate_discrete_histograms(
        processed,                         # (T, N) trajectory: rows are timesteps
        V=int(processed.max()) + 1,        # number of bins, must cover the largest count
        target=processed[-1],              # last timestep = target distribution
        center=s.center,                   # dashed line at the source center
        title_prefix=f"{s.process_type.upper()} Process applied to a {s.sample_type.upper()} Distribution (center {s.center})"
        )

### Discrete Flow Matching

In [8]:
%%time

# key is the dataset name, value is the dimension
data_dict = {"dpd":1,
             "catmoons": 2}
data = "dpd" #"catmoons" "dpd"
dim = data_dict[data]
V = 128
#set up the model and optimizer
# model = MLP(dim=dim, out_dim = dim * V, w=512, time_varying=True).to(device)
model = DiscreteAttentionFlow(
            D=dim,
            out_dim = dim * V,
            num_hidden=64,
            num_layers=3,
            num_heads=1,
            num_ffn_layers=2,
            num_conditions=None,
            activation=nn.GELU,
            dropout=0.05,
        ).to(device)

dpd_dataset = MeasureDataset(spec)
assert  type(dpd_dataset) == MeasureDataset, f"not a MeasureDataset object its a {type(dpd_dataset)}"
optimizer = torch.optim.Adam(model.parameters())
print(f"model on device: {device}")


dfm = DiscreteFlowMatching(optimizer, device, model, V, data, dataset=dpd_dataset)
# dfm = DiscreteFlowMatching(optimizer, device, model, V, "catmoons")

#training loop
dfm.train(epochs=20000, batch_size = 256)

/Users/magui/miniconda3/lib/python3.12/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


128 outdim
model on device: mps
Epoch:0: loss 4.852 time 0.73


Epoch:2000: loss 2.084 time 55.48


Epoch:4000: loss 3.061 time 56.21


Epoch:6000: loss 2.321 time 56.08


Epoch:8000: loss 3.121 time 55.60


... (3 outputs left out)

Epoch:16000: loss 2.832 time 54.94


Epoch:18000: loss 1.595 time 54.63


Epoch:19999: loss 0.738 time 53.16


CPU times: user 9min 8s, sys: 1min 32s, total: 10min 41s
Wall time: 9min 18s


In [10]:
SEED = 10
N = 1500
rng = np.random.default_rng(SEED)
CENTERS = [5]
N_MEASURES = 2
#initialize the data object using the list of centers
spec_test = make_measure_specs(centers=CENTERS, seed=SEED)

#initialize the data object using the list of centers
# spec = make_measure_specs(n_measures=N_MEASURES, seed=SEED)

#create the source and target according to the spec parameters, then plot
for s in spec_test:
    #processed is the timesteps of the discrete process
    processed_test = generate_target_dist(spec=s, sample_size=N, rng=rng)

    # plot_dpd(processed, sample_type=s.sample_type, process_type=s.process_type, center=s.center)

    animate_discrete_histograms(
        processed_test,                         # (T, N) trajectory: rows are timesteps
        V=int(processed_test.max()) + 1,        # number of bins, must cover the largest count
        target=processed_test[-1],              # last timestep = target distribution
        center=s.center,                   # dashed line at the source center
        title_prefix=f"{s.process_type.upper()} Process applied to a {s.sample_type.upper()} Distribution (center {s.center})"
        )

In [ ]:
dpd_test = MeasureDataset(spec_test)

dfm.dataset = dpd_test

traj = dfm.sample(return_trajectory=True)
traj = torch.stack(traj).cpu()
traj = traj.squeeze(-1) #drop D dimension (==1)
# animate_discrete_histograms(traj)
animate_discrete_histograms(
        traj[:,0],                         # (T, N) trajectory: rows are timesteps
        V=dfm.v,        # number of bins, must cover the largest count
        target=traj[-1],              # last timestep = target distribution
        center=spec_test[0].center,                   # dashed line at the source center
        title_prefix=f"{spec_test[0].process_type.upper()} Process applied to a {spec_test[0].sample_type.upper()} Distribution (center {spec_test[0].center})"
        )

### OG Conditional Flow Matching

In [ ]:
%%time
#2D distribution so dim is 2
dim = 2
#set up the model and optimizer
model = MLP(dim=dim, time_varying=True).to(device)
optimizer = torch.optim.Adam(model.parameters())
print(f"model on device: {device}")

fm = FlowMatching(optimizer, device, model)
#training loop
fm.train()